In [7]:

from pyspark.sql import SparkSession

DEFAULT_SPARK_REMOTE = "sc://localhost:15002"
spark = SparkSession.builder.remote(DEFAULT_SPARK_REMOTE).getOrCreate()

spark.version

'4.1.2'

In [2]:
spark.sql("use spark_catalog")
spark.sql("use schema northwind_db")
spark.sql("show tables").show()

+------------+-------------+-----------+
|   namespace|    tableName|isTemporary|
+------------+-------------+-----------+
|northwind_db|    employees|      false|
|northwind_db|order_details|      false|
|northwind_db|       orders|      false|
+------------+-------------+-----------+



In [3]:
from pyspark.sql.window import Window
from pyspark.sql.functions import min, col, datediff

windowSpec = Window.partitionBy("CustomerID").orderBy("OrderDate")

df = (spark.table("orders")
      .select("CustomerID", "OrderDate")
      .withColumn("FirstOrderDate", min(col("OrderDate")).over(windowSpec))
      .sort("CustomerID", "OrderDate"))

df.limit(10).show()


+----------+----------+--------------+
|CustomerID| OrderDate|FirstOrderDate|
+----------+----------+--------------+
|     ALFKI|1997-08-25|    1997-08-25|
|     ALFKI|1997-10-03|    1997-08-25|
|     ALFKI|1997-10-13|    1997-08-25|
|     ALFKI|1998-01-15|    1997-08-25|
|     ALFKI|1998-03-16|    1997-08-25|
|     ALFKI|1998-04-09|    1997-08-25|
|     ANATR|1996-09-18|    1996-09-18|
|     ANATR|1997-08-08|    1996-09-18|
|     ANATR|1997-11-28|    1996-09-18|
|     ANATR|1998-03-04|    1996-09-18|
+----------+----------+--------------+



In [4]:
spark.sql(""" 
    
    select 
        CustomerID,
        OrderDate,
        min(OrderDate) over (
                              partition by CustomerID
                              order by OrderDate
                            ) as FirstOrderDate
    from orders
    order by CustomerID, OrderDate
    
    """).limit(10).show()

+----------+----------+--------------+
|CustomerID| OrderDate|FirstOrderDate|
+----------+----------+--------------+
|     ALFKI|1997-08-25|    1997-08-25|
|     ALFKI|1997-10-03|    1997-08-25|
|     ALFKI|1997-10-13|    1997-08-25|
|     ALFKI|1998-01-15|    1997-08-25|
|     ALFKI|1998-03-16|    1997-08-25|
|     ALFKI|1998-04-09|    1997-08-25|
|     ANATR|1996-09-18|    1996-09-18|
|     ANATR|1997-08-08|    1996-09-18|
|     ANATR|1997-11-28|    1996-09-18|
|     ANATR|1998-03-04|    1996-09-18|
+----------+----------+--------------+



Calculate:
Revenue for each Order
Revenue for each customer
Running revenue for each customer over the time
Total running revenue over the time
Tatal revenue

In [5]:
spark.sql("""
            with revenue_by_customer as (
            select
               o.OrderDate as OrderDate,
               o.OrderID as OrderID,
               o.CustomerID as CustomerID,
               sum(round(UnitPrice * Quantity * (1 - Discount),2)) as OrderRevenue
            from order_details od
            left join orders o on od.OrderID = o.OrderID
            group by o.OrderDate, o.OrderID, o.CustomerID
            )
            select 
               CustomerID,
               OrderID,
               OrderDate,
               OrderRevenue,
               sum(OrderRevenue) over (partition by CustomerID) as CustomerRevenue,
               sum(OrderRevenue) over (partition by CustomerID order by OrderDate, OrderID) as CustomerRunningRevenue,
               sum(OrderRevenue) over (order by OrderDate, OrderID rows between unbounded preceding and current row) as RunningTotalRevenue,
               sum(OrderRevenue) over () as TotalRevenue
            from revenue_by_customer
            order by 
               CustomerID,
               OrderDate,
               OrderID

         """).limit(20).show()



+----------+-------+----------+------------+---------------+----------------------+-------------------+------------+
|CustomerID|OrderID| OrderDate|OrderRevenue|CustomerRevenue|CustomerRunningRevenue|RunningTotalRevenue|TotalRevenue|
+----------+-------+----------+------------+---------------+----------------------+-------------------+------------+
|     ALFKI|  10643|1997-08-25|      814.50|        4273.00|                814.50|          579284.19|  1265793.29|
|     ALFKI|  10692|1997-10-03|      878.00|        4273.00|               1692.50|          659026.24|  1265793.29|
|     ALFKI|  10702|1997-10-13|      330.00|        4273.00|               2022.50|          676749.22|  1265793.29|
|     ALFKI|  10835|1998-01-15|      845.80|        4273.00|               2868.30|          871343.70|  1265793.29|
|     ALFKI|  10952|1998-03-16|      471.20|        4273.00|               3339.50|         1058216.16|  1265793.29|
|     ALFKI|  11011|1998-04-09|      933.50|        4273.00|    

In [6]:
from pyspark.sql import Window
from pyspark.sql.functions import col, round, expr, sum

WindowCustomerRevenue = Window.partitionBy("CustomerID")
WindowCustomerRunningRevenue = Window.partitionBy("CustomerID").orderBy("OrderDate", "OrderID").rowsBetween(Window.unboundedPreceding, Window.currentRow)
WindowRunningTotalRevenue = Window.orderBy("OrderDate", "OrderID").rowsBetween(Window.unboundedPreceding, Window.currentRow)
WindowTotalRevenue = Window.rowsBetween(Window.unboundedPreceding, Window.unboundedFollowing)

df = (spark.table("order_details")
      .join(spark.table("orders"), "OrderID")
      .groupBy(col("CustomerID"), 
               col("OrderID"), 
               col("OrderDate"))
      .agg(sum(col("UnitPrice") * col("Quantity") * (1 - col("Discount"))).alias("OrderRevenue"))
      .withColumn("CustomerRevenue", sum(col("OrderRevenue")).over(WindowCustomerRevenue))
      .withColumn("CustomerRunningRevenue", sum(col("OrderRevenue")).over(WindowCustomerRunningRevenue))
      .withColumn("RunningTotalRevenue", sum(col("OrderRevenue")).over(WindowRunningTotalRevenue))
      .withColumn("TotalRevenue", sum(col("OrderRevenue")).over(WindowTotalRevenue))
      .sort("CustomerID", "OrderDate", "OrderID")
)

df.show()

/Users/talib/.pyenv/versions/3.10.12/lib/python3.10/site-packages/pyspark/sql/connect/expressions.py:945: UserWarning: WARN WindowExpression: No Partition Defined for Window operation! Moving all data to a single partition, this can cause serious performance degradation.
  warnings.warn(


+----------+-------+----------+------------+---------------+----------------------+-------------------+------------+
|CustomerID|OrderID| OrderDate|OrderRevenue|CustomerRevenue|CustomerRunningRevenue|RunningTotalRevenue|TotalRevenue|
+----------+-------+----------+------------+---------------+----------------------+-------------------+------------+
|     ALFKI|  10643|1997-08-25|    814.5000|      4273.0000|              814.5000|        579284.0925|1265793.0395|
|     ALFKI|  10692|1997-10-03|    878.0000|      4273.0000|             1692.5000|        659026.1100|1265793.0395|
|     ALFKI|  10702|1997-10-13|    330.0000|      4273.0000|             2022.5000|        676749.0785|1265793.0395|
|     ALFKI|  10835|1998-01-15|    845.8000|      4273.0000|             2868.3000|        871343.5325|1265793.0395|
|     ALFKI|  10952|1998-03-16|    471.2000|      4273.0000|             3339.5000|       1058215.9465|1265793.0395|
|     ALFKI|  11011|1998-04-09|    933.5000|      4273.0000|    